## 1. Project Setup and Data Paths

This notebook implements the semantic retrieval component of the proposed
query-conditioned recommendation system.

The retrieval pipeline follows a BLAIR-style semantic product search approach:

1. Represent each product in the Amazon Electronics catalogue using a semantic encoder.
2. Represent the user's synthetic query using the same encoder.
3. Retrieve products by comparing the query representation against the product representations using FAISS.
4. For the proposed personalised method, concatenate the user's preference profile with the synthetic query before encoding it.
5. Compare the personalised retrieval approach against query-only semantic retrieval.

The same semantic encoder and product index will be used for both methods to ensure
that the comparison isolates the effect of incorporating user preference information.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import faiss

from transformers import AutoTokenizer, AutoModel

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

/users/adhp511/.pyenv/versions/thesis/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.13.0+cu130
CUDA available: False


In [2]:
# Project and data paths

PROJECT_ROOT = Path("/users/adhp511/query-conditioned-recommender")

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
GENERATED_DIR = DATA_DIR / "generated"
MODELS_DIR = Path.home() / "models"

print("Project root:", PROJECT_ROOT)
print("Processed data:", PROCESSED_DIR)
print("Generated data:", GENERATED_DIR)
print("Local models:", MODELS_DIR)

print("\nProject root exists:", PROJECT_ROOT.exists())
print("Processed directory exists:", PROCESSED_DIR.exists())
print("Generated directory exists:", GENERATED_DIR.exists())
print("Models directory exists:", MODELS_DIR.exists())

Project root: /users/adhp511/query-conditioned-recommender
Processed data: /users/adhp511/query-conditioned-recommender/data/processed
Generated data: /users/adhp511/query-conditioned-recommender/data/generated
Local models: /users/adhp511/models

Project root exists: True
Processed directory exists: True
Generated directory exists: True
Models directory exists: True


## 2. Inspect Available Product Data

The retrieval system should be evaluated against the full Amazon Electronics
product catalogue available for this experiment.

Before constructing the FAISS index, we inspect the generated and processed
data directories to identify the available product-level datasets and their
schemas.

The previously used candidate subset is not assumed to be the final retrieval
catalogue. The catalogue used here will be verified explicitly before
embedding and indexing.

In [3]:
# Inspect available data files

print("=== Processed data ===")
for path in sorted(PROCESSED_DIR.iterdir()):
    print(path.name)

print("\n=== Generated data ===")
for path in sorted(GENERATED_DIR.iterdir()):
    print(path.name)

=== Processed data ===
enriched_user_histories.parquet
filtered_reviews.parquet
user_histories.parquet
user_preference_profiles.parquet

=== Generated data ===
aligned_evaluation_data.parquet
baselines
bge_baseline_results.parquet
bge_baseline_retrievals.parquet
bm25_predictions.parquet
candidate_metadata.parquet
evaluation_set_144.csv
faiss_product_index.bin
product_embeddings.npy
proposed_model_data.parquet
query_constraints_qwen.parquet
query_constraints_qwen_checkpoint.parquet
synthetic_queries.parquet
synthetic_queries_checkpoint.parquet


## 3. Locate the Full Amazon Electronics Product Catalogue

The previous retrieval pipeline used a restricted candidate catalogue
(`candidate_metadata.parquet`). For the revised semantic retrieval system,
the product search space should instead correspond to the full Amazon
Electronics catalogue available to the experiment.

We therefore inspect the project directory for other product-level data files
before constructing the new product embeddings or FAISS index.

In [4]:
# Search the project for product/catalogue data files
catalogue_candidates = []

for path in PROJECT_ROOT.rglob("*"):
    if path.is_file():
        name = path.name.lower()

        if any(keyword in name for keyword in [
            "metadata",
            "product",
            "catalog",
            "electronics"
        ]):
            catalogue_candidates.append(path)

for path in sorted(catalogue_candidates):
    try:
        size_mb = path.stat().st_size / (1024 ** 2)
        print(f"{path.relative_to(PROJECT_ROOT)}  |  {size_mb:.2f} MB")
    except Exception:
        print(path.relative_to(PROJECT_ROOT))

data/generated/candidate_metadata.parquet  |  23.16 MB
data/generated/faiss_product_index.bin  |  19.29 MB
data/generated/product_embeddings.npy  |  19.29 MB
data/raw/Electronics.jsonl  |  21568.52 MB
data/raw/meta_Electronics.jsonl  |  5003.11 MB
notebooks/03_metadata_enrichment.ipynb  |  0.04 MB
scripts/join_metadata.py  |  0.00 MB
venv/lib/python3.9/site-packages/pip/_internal/operations/build/__pycache__/metadata.cpython-39.pyc  |  0.00 MB
venv/lib/python3.9/site-packages/pip/_internal/operations/build/__pycache__/metadata_editable.cpython-39.pyc  |  0.00 MB
venv/lib/python3.9/site-packages/pip/_internal/operations/build/__pycache__/metadata_legacy.cpython-39.pyc  |  0.00 MB
venv/lib/python3.9/site-packages/pip/_internal/operations/build/metadata.py  |  0.00 MB
venv/lib/python3.9/site-packages/pip/_internal/operations/build/metadata_editable.py  |  0.00 MB
venv/lib/python3.9/site-packages/pip/_internal/operations/build/metadata_legacy.py  |  0.00 MB
venv/lib/python3.9/site-packages

## 4. Inspect the Amazon Electronics Metadata

The full Amazon Electronics product catalogue is stored in
`meta_Electronics.jsonl`.

We first inspect a small number of records to determine the available product
fields and identify the metadata that will be used to construct semantic
product representations.

Following the BLAIR-style semantic retrieval setup, product representations
will be created from textual product information and encoded using the same
semantic encoder used for query representations.

In [5]:
# Inspect the first few product metadata records

META_PATH = DATA_DIR / "raw" / "meta_Electronics.jsonl"

print("Metadata file:", META_PATH)
print("Exists:", META_PATH.exists())
print("Size:", f"{META_PATH.stat().st_size / (1024**3):.2f} GB")

with open(META_PATH, "r", encoding="utf-8") as f:
    for i in range(3):
        line = f.readline()
        print(f"\n--- Record {i + 1} ---")
        print(line[:3000])

Metadata file: /users/adhp511/query-conditioned-recommender/data/raw/meta_Electronics.jsonl
Exists: True
Size: 4.89 GB

--- Record 1 ---
{"main_category": "All Electronics", "title": "FS-1051 FATSHARK TELEPORTER V3 HEADSET", "average_rating": 3.5, "rating_number": 6, "features": [], "description": ["Teleporter V3 The \u201cTeleporter V3\u201d kit sets a new level of value in the FPV world with Fat Shark renowned performance and quality. The fun of FPV is experienced firsthand through the large screen FPV headset with integrated NexwaveRF receiver technology while simultaneously recording onboard HD footage with the included \u201cPilotHD\u201d camera. The \u201cTeleporter V3\u201d kit comes complete with everything you need to step into the cockpit of your FPV vehicle. We\u2019ve included our powerful 250mW 5.8Ghz transmitter, 25 degree FOV headset (largest QVGA display available), the brand new \u201cPilotHD\u201d camera with live AV out and all the cables, antennas and connectors nee

## 5. Define the Product Representation

Each product will be represented using concise textual metadata:

- Product title
- Store/brand
- Product categories

Fields such as images, videos, ratings, prices, and detailed product
attributes will not be included in the semantic representation.

This provides a consistent textual representation for the full catalogue while
avoiding unnecessary metadata that may introduce noise into semantic retrieval.

The product identifier (`parent_asin`) will be retained separately so that
retrieval results can be mapped back to the original Amazon products.

In [6]:
# Define how a raw Amazon metadata record will be converted
# into the textual representation used for semantic encoding.

import json


def build_product_text(record):
    title = str(record.get("title") or "").strip()
    store = str(record.get("store") or "").strip()

    categories = record.get("categories") or []
    categories = [str(x).strip() for x in categories if str(x).strip()]

    parts = []

    if title:
        parts.append(f"Title: {title}")

    if store:
        parts.append(f"Brand: {store}")

    if categories:
        parts.append(f"Category: {' > '.join(categories)}")

    return " | ".join(parts)


# Test the representation on the first record
with open(META_PATH, "r", encoding="utf-8") as f:
    example_record = json.loads(f.readline())

print(build_product_text(example_record))

Title: FS-1051 FATSHARK TELEPORTER V3 HEADSET | Brand: Fat Shark | Category: Electronics > Television & Video > Video Glasses


## 6. Validate the Full Product Catalogue

Before generating embeddings, we validate the catalogue at the product level.

The metadata file is processed line-by-line to avoid loading the approximately
5 GB JSONL file into memory. We count:

- Total metadata records
- Records containing a `parent_asin`
- Unique `parent_asin` identifiers
- Duplicate metadata records based on `parent_asin`

The unique `parent_asin` count will determine the final number of product
vectors in the FAISS index.

In [7]:
# Validate catalogue size and product identifier uniqueness

total_records = 0
records_with_asin = 0
parent_asins = set()

with open(META_PATH, "r", encoding="utf-8") as f:
    for line in f:
        total_records += 1

        record = json.loads(line)
        parent_asin = record.get("parent_asin")

        if parent_asin:
            records_with_asin += 1
            parent_asins.add(parent_asin)

print("Total metadata records:", f"{total_records:,}")
print("Records with parent_asin:", f"{records_with_asin:,}")
print("Unique parent_asin values:", f"{len(parent_asins):,}")
print("Duplicate records by parent_asin:", f"{records_with_asin - len(parent_asins):,}")

Total metadata records: 1,610,012
Records with parent_asin: 1,610,012
Unique parent_asin values: 1,610,012
Duplicate records by parent_asin: 0


## 7. Select the Semantic Encoder

The semantic retrieval baseline follows the BLAIR-style setup in which the same
semantic encoder is used to represent both queries and products.

The encoder must therefore produce compatible representations for:

- synthetic user queries in the query-only baseline;
- concatenated user-profile and query text in the proposed method; and
- product metadata in the Amazon Electronics catalogue.

The semantic encoder will be selected from the encoders evaluated in the BLAIR
paper rather than introducing an unrelated embedding model.

Before loading a model, we inspect the locally available model cache so that
the implementation remains compatible with the offline computing environment.

In [8]:
# Inspect locally available Hugging Face model directories

print("Local model directories:\n")

for path in sorted(MODELS_DIR.iterdir()):
    if path.is_dir():
        print(path.name)

Local model directories:

bge-base-en-v1.5


## 8. Verify Available Semantic Encoder Resources

The local model directory currently contains `bge-base-en-v1.5`.

BGE is not one of the semantic encoders evaluated in the BLAIR paper, so it
will not be adopted automatically for the final BLAIR-style comparison.

We therefore inspect the Hugging Face cache available to the current
environment for models corresponding to the semantic encoders evaluated in
BLAIR. If no suitable encoder is available locally, the final encoder choice
will need to be made explicitly rather than silently substituting BGE.

In [9]:
# Inspect the local Hugging Face cache for available model snapshots

HF_CACHE = Path.home() / ".cache" / "huggingface" / "hub"

print("Hugging Face cache:", HF_CACHE)
print("Exists:", HF_CACHE.exists())

if HF_CACHE.exists():
    print("\nCached model repositories:\n")
    
    for path in sorted(HF_CACHE.glob("models--*")):
        print(path.name)

Hugging Face cache: /users/adhp511/.cache/huggingface/hub
Exists: True

Cached model repositories:

models--Qwen--Qwen2.5-7B-Instruct
models--distilgpt2


## 9. Check Installed Transformer Models and Packages

The Hugging Face cache does not currently contain any of the semantic encoders
used in the BLAIR evaluation.

Before making a final encoder choice, we check the current Python environment
for installed libraries and locally available model resources that could
provide one of the BLAIR-evaluated encoders.

No model will be downloaded at this stage.

In [10]:
# Check relevant installed packages

import importlib.util

packages_to_check = [
    "sentence_transformers",
    "transformers",
    "peft",
    "accelerate",
    "faiss",
]

for package in packages_to_check:
    available = importlib.util.find_spec(package) is not None
    print(f"{package:<20} {'available' if available else 'NOT available'}")

sentence_transformers NOT available
transformers         available
peft                 NOT available
accelerate           available
faiss                available


## 10. Semantic Encoder for Retrieval

The BLAIR paper evaluates multiple semantic encoders for zero-shot product
search, including E5 and Sentence-T5.

The current Hyperion environment does not contain a cached BLAIR semantic
encoder, and `sentence-transformers` is not installed. Therefore, the
retrieval implementation will use a BLAIR-evaluated encoder that can be
loaded directly through the Hugging Face Transformers interface.

For this implementation, **E5** is selected as the semantic encoder.

The same E5 encoder will be used for:

- product representations;
- query-only representations for the BLAIR-style baseline; and
- profile + query representations for the proposed personalised method.

No separate query/product encoder is introduced.

The encoder will be used to produce dense representations, while FAISS will
be used only for efficient nearest-neighbour search over the product
representations.

In [11]:
# Check whether the selected E5 model is already available locally

import os

os.environ["HF_HUB_OFFLINE"] = "1"

E5_MODEL_NAME = "intfloat/e5-base-v2"

try:
    tokenizer = AutoTokenizer.from_pretrained(
        E5_MODEL_NAME,
        local_files_only=True
    )
    
    model = AutoModel.from_pretrained(
        E5_MODEL_NAME,
        local_files_only=True
    )
    
    print("E5 model is available locally.")
    
except Exception as e:
    print("E5 model is NOT available locally.")
    print("\nReason:")
    print(type(e).__name__, str(e)[:1000])

E5 model is NOT available locally.

Reason:
OSError We couldn't connect to 'https://huggingface.co' to load the files, and couldn't find them in the cached files.
Check your internet connection or see how to run the library in offline mode at 'https://huggingface.co/docs/transformers/installation#offline-mode'.


## 12. Load the E5 Semantic Encoder

The E5 model has been transferred to the Hyperion environment and will be
loaded entirely from the local filesystem.

`local_files_only=True` is used to ensure that the experiment does not depend
on network connectivity during execution.

The model will be moved to the available GPU for semantic encoding.

In [12]:
#Load E5 from the local model directory
E5_MODEL_PATH= MODELS_DIR/"e5-base-v2"
device= torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("E5 model path:", E5_MODEL_PATH)
print("Model directory exists:", E5_MODEL_PATH.exists())
print("Device:", device)
e5_tokenizer= AutoTokenizer.from_pretrained(E5_MODEL_PATH,local_files_only=True)
e5_model= AutoModel.from_pretrained(E5_MODEL_PATH,local_files_only=True)
e5_model= e5_model.to(device)
e5_model.eval()
print("\nE5 model loaded successfully.")
print("Model hidden size:", e5_model.config.hidden_size)
print("Model device:", next(e5_model.parameters()).device)

E5 model path: /users/adhp511/models/e5-base-v2
Model directory exists: True
Device: cuda


Loading weights: 100%|██████████████████████████████████████████████| 199/199 [00:00<00:00, 1266.27it/s]



E5 model loaded successfully.
Model hidden size: 768
Model device: cuda:0


## 13. Test E5 Query Encoding

E5 uses task-specific prefixes to distinguish retrieval queries from
documents. For semantic retrieval, query text is prefixed with `query:`.

We first encode a single synthetic-style user query to verify that:

- tokenisation works correctly;
- inference runs on the GPU;
- the encoder produces a 768-dimensional representation; and
- the representation can be L2-normalised for cosine-similarity retrieval
  with FAISS.

In [13]:
#Test E5 query encoding with a single example
test_query = "query: looking for wireless earbuds with good sound quality and long battery life"
inputs= e5_tokenizer(test_query,return_tensors="pt",truncation=True,max_length=512)
inputs= {key: value.to(device) for key, value in inputs.items()}
with torch.no_grad():
    outputs= e5_model(**inputs)
#E5 uses mean pooling over the token representations
attention_mask= inputs["attention_mask"].unsqueeze(-1)
token_embeddings= outputs.last_hidden_state
pooled_embedding= (token_embeddings*attention_mask).sum(dim=1)/attention_mask.sum(dim=1)
#L2 normalisation for cosine similarity
query_embedding= torch.nn.functional.normalize(pooled_embedding,p=2,dim=1)
query_embedding= query_embedding.cpu().numpy().astype("float32")
print("Embedding shape:", query_embedding.shape)
print("Embedding dtype:", query_embedding.dtype)
print("L2 norm:", np.linalg.norm(query_embedding[0]))
print("First 10 values:", query_embedding[0][:10])

Embedding shape: (1, 768)
Embedding dtype: float32
L2 norm: 1.0
First 10 values: [-0.00063319 -0.02554371 -0.053884   -0.00544581  0.04844724 -0.02832565
  0.01039235  0.00164622  0.00674745 -0.02788564]


## 14. Prepare the Full Product Catalogue for Encoding

The full Amazon Electronics metadata contains 1,610,012 unique products.

Products will be processed in batches to avoid loading the complete 5 GB metadata
file into memory.

For each product, we retain:

- `parent_asin` — the unique Amazon product identifier;
- `title` — the product title;
- `store` — the available brand/store field;
- `categories` — the product category hierarchy; and
- `product_text` — the textual representation used by E5.

The product records and embeddings will preserve the same ordering. This allows
a FAISS result index to be mapped directly back to the corresponding
`parent_asin`.

Product text will be encoded using the E5 `passage:` prefix because products
act as the retrieval corpus, while user queries will use the `query:` prefix.

In [14]:
# Define output paths and encoding parameters

PRODUCT_CATALOGUE_PATH = GENERATED_DIR / "full_product_catalogue.parquet"
PRODUCT_EMBEDDINGS_PATH = GENERATED_DIR / "e5_product_embeddings.npy"
FAISS_INDEX_PATH = GENERATED_DIR / "e5_faiss_product_index.bin"

EMBEDDING_DIM = e5_model.config.hidden_size

# Start conservatively; we will benchmark this before processing 1.61M products.
BATCH_SIZE = 64
MAX_LENGTH = 256

print("Catalogue output:", PRODUCT_CATALOGUE_PATH)
print("Embeddings output:", PRODUCT_EMBEDDINGS_PATH)
print("FAISS index output:", FAISS_INDEX_PATH)
print("Embedding dimension:", EMBEDDING_DIM)
print("Batch size:", BATCH_SIZE)
print("Maximum token length:", MAX_LENGTH)

Catalogue output: /users/adhp511/query-conditioned-recommender/data/generated/full_product_catalogue.parquet
Embeddings output: /users/adhp511/query-conditioned-recommender/data/generated/e5_product_embeddings.npy
FAISS index output: /users/adhp511/query-conditioned-recommender/data/generated/e5_faiss_product_index.bin
Embedding dimension: 768
Batch size: 64
Maximum token length: 256


## 15. Benchmark Product Encoding

A small batch of real Amazon Electronics products is encoded before starting
the full catalogue job.

This benchmark is used to:

- verify the product-text construction;
- verify E5 `passage:` encoding;
- confirm GPU inference;
- measure embedding throughput; and
- estimate the approximate runtime for the complete 1,610,012-product catalogue.

The benchmark does not create or modify the final catalogue or FAISS index.

In [15]:
# Read and encode a small batch of real catalogue products

import time

benchmark_records = []

with open(META_PATH, "r", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)

        if record.get("parent_asin"):
            benchmark_records.append({
                "parent_asin": record["parent_asin"],
                "title": record.get("title") or "",
                "store": record.get("store") or "",
                "categories": record.get("categories") or [],
                "product_text": build_product_text(record),
            })

        if len(benchmark_records) >= BATCH_SIZE:
            break

benchmark_df = pd.DataFrame(benchmark_records)

product_texts = [
    f"passage: {text}"
    for text in benchmark_df["product_text"].tolist()
]

print("Benchmark products:", len(product_texts))
print("\nFirst product:")
print(benchmark_df.iloc[0]["product_text"])

# Tokenisation
inputs = e5_tokenizer(
    product_texts,
    padding=True,
    truncation=True,
    max_length=MAX_LENGTH,
    return_tensors="pt"
)

inputs = {
    key: value.to(device)
    for key, value in inputs.items()
}

# GPU synchronisation before timing
if torch.cuda.is_available():
    torch.cuda.synchronize()

start_time = time.perf_counter()

with torch.no_grad():
    outputs = e5_model(**inputs)

attention_mask = inputs["attention_mask"].unsqueeze(-1)

token_embeddings = outputs.last_hidden_state

pooled_embeddings = (
    token_embeddings * attention_mask
).sum(dim=1) / attention_mask.sum(dim=1)

product_embeddings_test = torch.nn.functional.normalize(
    pooled_embeddings,
    p=2,
    dim=1
)

if torch.cuda.is_available():
    torch.cuda.synchronize()

elapsed = time.perf_counter() - start_time

product_embeddings_test = (
    product_embeddings_test
    .cpu()
    .numpy()
    .astype("float32")
)

print("\nEmbedding shape:", product_embeddings_test.shape)
print("Embedding dtype:", product_embeddings_test.dtype)
print("Average L2 norm:", np.linalg.norm(product_embeddings_test, axis=1).mean())
print("Encoding time:", f"{elapsed:.2f} seconds")
print("Throughput:", f"{len(product_texts) / elapsed:.2f} products/sec")

Benchmark products: 64

First product:
Title: FS-1051 FATSHARK TELEPORTER V3 HEADSET | Brand: Fat Shark | Category: Electronics > Television & Video > Video Glasses

Embedding shape: (64, 768)
Embedding dtype: float32
Average L2 norm: 1.0
Encoding time: 0.19 seconds
Throughput: 331.39 products/sec


## 16. Plan the Full Catalogue Encoding

The benchmark demonstrates that the RTX 8000 can encode the product
representations efficiently.

The complete catalogue will be encoded in batches and stored as a NumPy
memory-mapped array rather than held entirely in GPU or system memory.

A memory-mapped array is used so that:

- embeddings can be written incrementally;
- the complete 1,610,012 × 768 float32 matrix does not need to reside in RAM;
- the encoding process can be resumed if the GPU job is interrupted; and
- the resulting matrix can later be supplied directly to FAISS.

The expected embedding storage is approximately 4.6 GB:

1,610,012 products × 768 dimensions × 4 bytes per float32 value.

In [16]:
# Calculate expected storage and rough encoding time

N_PRODUCTS = 1_610_012

BYTES_PER_EMBEDDING = EMBEDDING_DIM * np.dtype("float32").itemsize
TOTAL_BYTES = N_PRODUCTS * BYTES_PER_EMBEDDING

BENCHMARK_THROUGHPUT = 331.39

ESTIMATED_ENCODING_SECONDS = N_PRODUCTS / BENCHMARK_THROUGHPUT

print("Products:", f"{N_PRODUCTS:,}")
print("Embedding dimensions:", EMBEDDING_DIM)
print("Embedding dtype: float32")
print("Expected embedding size:", f"{TOTAL_BYTES / (1024**3):.2f} GB")
print(
    "Estimated GPU encoding time:",
    f"{ESTIMATED_ENCODING_SECONDS / 3600:.2f} hours"
)

Products: 1,610,012
Embedding dimensions: 768
Embedding dtype: float32
Expected embedding size: 4.61 GB
Estimated GPU encoding time: 1.35 hours


## 17. Initialise the Product Embedding Matrix

A memory-mapped NumPy array will be used to store the complete product
embedding matrix.

The matrix has one row per product and 768 float32 dimensions. Its row order
will exactly match the product order in the catalogue.

The file is created on disk and populated incrementally during catalogue
encoding. This avoids requiring approximately 4.61 GB of continuous RAM.

If the encoding process is interrupted, the existing embedding file can be
reused rather than starting from an empty matrix.

In [17]:
# Initialise a memory-mapped embedding matrix

EXPECTED_EMBEDDING_SHAPE = (N_PRODUCTS, EMBEDDING_DIM)

if PRODUCT_EMBEDDINGS_PATH.exists():
    existing_size = PRODUCT_EMBEDDINGS_PATH.stat().st_size
    expected_size = (
        N_PRODUCTS
        * EMBEDDING_DIM
        * np.dtype("float32").itemsize
    )

    print("Existing embedding file found.")
    print("Existing size:", f"{existing_size / (1024**3):.2f} GB")
    print("Expected size:", f"{expected_size / (1024**3):.2f} GB")

    if existing_size != expected_size:
        raise ValueError(
            "Existing embedding file has an unexpected size. "
            "Delete or rename it before continuing."
        )

    product_embeddings = np.memmap(
        PRODUCT_EMBEDDINGS_PATH,
        dtype="float32",
        mode="r+",
        shape=EXPECTED_EMBEDDING_SHAPE
    )

    print("Existing embedding matrix opened for continuation.")

else:
    product_embeddings = np.memmap(
        PRODUCT_EMBEDDINGS_PATH,
        dtype="float32",
        mode="w+",
        shape=EXPECTED_EMBEDDING_SHAPE
    )

    product_embeddings[:] = 0.0
    product_embeddings.flush()

    print("New embedding matrix created.")

print("Shape:", product_embeddings.shape)
print("Dtype:", product_embeddings.dtype)
print("File:", PRODUCT_EMBEDDINGS_PATH)

New embedding matrix created.
Shape: (1610012, 768)
Dtype: float32
File: /users/adhp511/query-conditioned-recommender/data/generated/e5_product_embeddings.npy


## 18. Build the Full Product Catalogue Metadata

The raw Amazon Electronics metadata is processed sequentially and converted
into a compact product catalogue.

Only the metadata required for retrieval and result interpretation is retained:

- `parent_asin`
- `title`
- `store`
- `categories`
- `product_text`

The catalogue is written in the same order as the source metadata. This order
will be used as the canonical row order for the product embedding matrix and
FAISS index.

The catalogue is written using PyArrow's Parquet writer so that the complete
dataset can be written incrementally without loading all 1.61 million products
into memory.

In [21]:
# Build the complete compact product catalogue incrementally
import pyarrow as pa
import pyarrow.parquet as pq

if PRODUCT_CATALOGUE_PATH.exists():
    PRODUCT_CATALOGUE_PATH.unlink()
    print("Removed incomplete catalogue.")

schema = pa.schema([
    ("parent_asin", pa.string()),
    ("title", pa.string()),
    ("store", pa.string()),
    ("categories", pa.list_(pa.string())),
    ("product_text", pa.string()),
])

writer = pq.ParquetWriter(
    PRODUCT_CATALOGUE_PATH,
    schema=schema,
    compression="snappy"
)

CHUNK_SIZE = 10_000
chunk_records = []
total_written = 0

try:
    with open(META_PATH, "r", encoding="utf-8") as f:

        for line in f:
            record = json.loads(line)

            parent_asin = record.get("parent_asin")

            if not parent_asin:
                continue

            categories = record.get("categories") or []
            categories = [
                str(category).strip()
                for category in categories
                if str(category).strip()
            ]

            chunk_records.append({
                "parent_asin": str(parent_asin),
                "title": str(record.get("title") or ""),
                "store": str(record.get("store") or ""),
                "categories": categories,
                "product_text": build_product_text(record),
            })

            if len(chunk_records) >= CHUNK_SIZE:

                chunk_table = pa.Table.from_pylist(
                    chunk_records,
                    schema=schema
                )

                writer.write_table(chunk_table)

                total_written += len(chunk_records)
                chunk_records = []

                if total_written % 100_000 == 0:
                    print(f"Products written: {total_written:,}")

    # Write remaining records
    if chunk_records:
        chunk_table = pa.Table.from_pylist(
            chunk_records,
            schema=schema
        )

        writer.write_table(chunk_table)
        total_written += len(chunk_records)

finally:
    writer.close()

print("\nCatalogue creation complete.")
print("Products written:", f"{total_written:,}")
print("Expected products:", f"{N_PRODUCTS:,}")
print("Catalogue exists:", PRODUCT_CATALOGUE_PATH.exists())
print(
    "Catalogue size:",
    f"{PRODUCT_CATALOGUE_PATH.stat().st_size / (1024**2):.2f} MB"
)

Removed incomplete catalogue.
Products written: 100,000
Products written: 200,000
Products written: 300,000
Products written: 400,000
Products written: 500,000
Products written: 600,000
Products written: 700,000
Products written: 800,000
Products written: 900,000
Products written: 1,000,000
Products written: 1,100,000
Products written: 1,200,000
Products written: 1,300,000
Products written: 1,400,000
Products written: 1,500,000
Products written: 1,600,000

Catalogue creation complete.
Products written: 1,610,012
Expected products: 1,610,012
Catalogue exists: True
Catalogue size: 343.43 MB


## 19. Encode the Full Product Catalogue

The complete product catalogue is now available in Parquet format.

Each product will be converted to an E5 `passage:` representation and encoded
using the GPU in batches.

The resulting 768-dimensional, L2-normalised embeddings are written directly
to the memory-mapped embedding matrix created earlier.

The encoding process records progress in a checkpoint file. If execution is
interrupted, the notebook can resume from the last completed batch rather than
re-encoding products that have already been processed.

The product ordering remains unchanged throughout the process, preserving the
one-to-one mapping between catalogue rows and embedding rows.

In [22]:
# Check the final catalogue before starting GPU encoding

catalogue_check = pd.read_parquet(
    PRODUCT_CATALOGUE_PATH,
    columns=["parent_asin", "product_text"]
)

print("Catalogue shape:", catalogue_check.shape)
print("Columns:", catalogue_check.columns.tolist())
print("Missing parent_asin:", catalogue_check["parent_asin"].isna().sum())
print("Missing product_text:", catalogue_check["product_text"].isna().sum())
print("Unique parent_asin:", catalogue_check["parent_asin"].nunique())

print("\nFirst product:")
print(catalogue_check.iloc[0]["product_text"])

del catalogue_check

Catalogue shape: (1610012, 2)
Columns: ['parent_asin', 'product_text']
Missing parent_asin: 0
Missing product_text: 0
Unique parent_asin: 1610012

First product:
Title: FS-1051 FATSHARK TELEPORTER V3 HEADSET | Brand: Fat Shark | Category: Electronics > Television & Video > Video Glasses


## 20. Initialise the Encoding Checkpoint

The full catalogue contains 1,610,012 products, so encoding is a long-running
operation.

A small JSON checkpoint records the number of catalogue rows that have been
successfully written to the embedding matrix.

The checkpoint allows the encoding process to resume after an interruption
without re-encoding completed products.

The checkpoint is updated only after a batch has been successfully written and
flushed to disk.

In [23]:
# Initialise or recover the product-encoding checkpoint

CHECKPOINT_PATH = GENERATED_DIR / "e5_product_encoding_checkpoint.json"

if CHECKPOINT_PATH.exists():
    with open(CHECKPOINT_PATH, "r", encoding="utf-8") as f:
        checkpoint = json.load(f)

    start_row = int(checkpoint["completed_rows"])

    print("Existing checkpoint found.")
    print("Completed rows:", f"{start_row:,}")

else:
    checkpoint = {
        "completed_rows": 0,
        "total_rows": N_PRODUCTS,
        "embedding_dimension": EMBEDDING_DIM,
        "model": "intfloat/e5-base-v2",
        "batch_size": BATCH_SIZE,
        "max_length": MAX_LENGTH,
    }

    with open(CHECKPOINT_PATH, "w", encoding="utf-8") as f:
        json.dump(checkpoint, f, indent=2)

    start_row = 0

    print("New checkpoint created.")
    print("Starting from row:", start_row)

print("Checkpoint path:", CHECKPOINT_PATH)

New checkpoint created.
Starting from row: 0
Checkpoint path: /users/adhp511/query-conditioned-recommender/data/generated/e5_product_encoding_checkpoint.json


## 21. Test Streaming Product Encoding

Before encoding the complete catalogue, run a small batch through the exact
pipeline that will be used for the full run.

This test verifies:

1. Products can be streamed from the Parquet catalogue.
2. E5 receives the required `passage:` prefix for product text.
3. Embeddings have the expected dimensionality.
4. Embeddings are L2-normalised.
5. The memory-mapped embedding matrix accepts the output correctly.

No checkpoint progress is changed by this test.

In [26]:
# Configure the computation device and move E5 to it

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

e5_model = e5_model.to(DEVICE)
e5_model.eval()

EMBEDDING_DIM = e5_model.config.hidden_size

print("Device:", DEVICE)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("Embedding dimension:", EMBEDDING_DIM)
print("Model ready for inference.")

Device: cuda
GPU: Quadro RTX 8000
Embedding dimension: 768
Model ready for inference.


In [27]:
# Reload E5 tokenizer and model into the current kernel

E5_MODEL_PATH = Path.home() / "models" / "e5-base-v2"

print("Loading E5 from:", E5_MODEL_PATH)

tokenizer = AutoTokenizer.from_pretrained(
    E5_MODEL_PATH,
    local_files_only=True
)

e5_model = AutoModel.from_pretrained(
    E5_MODEL_PATH,
    local_files_only=True
)

e5_model = e5_model.to(DEVICE)
e5_model.eval()

EMBEDDING_DIM = e5_model.config.hidden_size

print("Model loaded successfully.")
print("Embedding dimension:", EMBEDDING_DIM)
print("Device:", DEVICE)
print("CUDA available:", torch.cuda.is_available())

Loading E5 from: /users/adhp511/models/e5-base-v2


Loading weights: 100%|██████████████████████████████████████████████| 199/199 [00:00<00:00, 7715.25it/s]

Model loaded successfully.
Embedding dimension: 768
Device: cuda
CUDA available: True


In [28]:
# Test one real catalogue batch through the E5 encoding pipeline

import pyarrow.parquet as pq

TEST_BATCH_SIZE = 64

parquet_file = pq.ParquetFile(PRODUCT_CATALOGUE_PATH)

test_batch = next(
    parquet_file.iter_batches(
        batch_size=TEST_BATCH_SIZE,
        columns=["product_text"]
    )
)

test_texts = test_batch.column("product_text").to_pylist()

print("Test products:", len(test_texts))
print("First product:")
print(test_texts[0])

# E5 requires the passage prefix for documents/products
test_inputs = [f"passage: {text}" for text in test_texts]

inputs = tokenizer(
    test_inputs,
    padding=True,
    truncation=True,
    max_length=MAX_LENGTH,
    return_tensors="pt"
)

inputs = {key: value.to(DEVICE) for key, value in inputs.items()}

with torch.no_grad():
    outputs = e5_model(**inputs)

attention_mask = inputs["attention_mask"].unsqueeze(-1)

masked_embeddings = outputs.last_hidden_state * attention_mask
sum_embeddings = masked_embeddings.sum(dim=1)
sum_mask = attention_mask.sum(dim=1).clamp(min=1)

test_embeddings = sum_embeddings / sum_mask
test_embeddings = torch.nn.functional.normalize(
    test_embeddings,
    p=2,
    dim=1
)

test_embeddings = test_embeddings.cpu().numpy().astype(np.float32)

print("\nEmbedding shape:", test_embeddings.shape)
print("Embedding dtype:", test_embeddings.dtype)
print("Average L2 norm:", np.linalg.norm(test_embeddings, axis=1).mean())
print("Minimum L2 norm:", np.linalg.norm(test_embeddings, axis=1).min())
print("Maximum L2 norm:", np.linalg.norm(test_embeddings, axis=1).max())

assert test_embeddings.shape == (TEST_BATCH_SIZE, EMBEDDING_DIM)
assert test_embeddings.dtype == np.float32
assert np.allclose(
    np.linalg.norm(test_embeddings, axis=1),
    1.0,
    atol=1e-5
)

print("\n✓ Streaming encoding test passed.")

Test products: 64
First product:
Title: FS-1051 FATSHARK TELEPORTER V3 HEADSET | Brand: Fat Shark | Category: Electronics > Television & Video > Video Glasses

Embedding shape: (64, 768)
Embedding dtype: float32
Average L2 norm: 1.0
Minimum L2 norm: 0.9999999
Maximum L2 norm: 1.0000001

✓ Streaming encoding test passed.


## 22. Benchmark a Larger Encoding Batch

The initial benchmark used a batch size of 64. Before the full catalogue
encoding, test a batch size of 256 to determine whether the GPU can process
larger batches efficiently.

If successful, the larger batch size will reduce tokenizer/model overhead
during the long-running catalogue encoding.

In [29]:
# Benchmark E5 encoding with a larger batch size

import time
import pyarrow.parquet as pq

BENCHMARK_BATCH_SIZE = 256

parquet_file = pq.ParquetFile(PRODUCT_CATALOGUE_PATH)

benchmark_batch = next(
    parquet_file.iter_batches(
        batch_size=BENCHMARK_BATCH_SIZE,
        columns=["product_text"]
    )
)

benchmark_texts = benchmark_batch.column("product_text").to_pylist()
benchmark_inputs = [f"passage: {text}" for text in benchmark_texts]

start_time = time.time()

inputs = tokenizer(
    benchmark_inputs,
    padding=True,
    truncation=True,
    max_length=MAX_LENGTH,
    return_tensors="pt"
)

inputs = {key: value.to(DEVICE) for key, value in inputs.items()}

with torch.no_grad():
    outputs = e5_model(**inputs)

attention_mask = inputs["attention_mask"].unsqueeze(-1)

masked_embeddings = outputs.last_hidden_state * attention_mask
sum_embeddings = masked_embeddings.sum(dim=1)
sum_mask = attention_mask.sum(dim=1).clamp(min=1)

benchmark_embeddings = sum_embeddings / sum_mask

benchmark_embeddings = torch.nn.functional.normalize(
    benchmark_embeddings,
    p=2,
    dim=1
)

benchmark_embeddings = benchmark_embeddings.cpu().numpy().astype(np.float32)

elapsed = time.time() - start_time
throughput = len(benchmark_texts) / elapsed

print("Batch size:", BENCHMARK_BATCH_SIZE)
print("Embedding shape:", benchmark_embeddings.shape)
print("Encoding time:", round(elapsed, 3), "seconds")
print("Throughput:", round(throughput, 2), "products/second")
print("Average L2 norm:", np.linalg.norm(benchmark_embeddings, axis=1).mean())

assert benchmark_embeddings.shape == (
    BENCHMARK_BATCH_SIZE,
    EMBEDDING_DIM
)

print("\n✓ Larger-batch benchmark passed.")

Batch size: 256
Embedding shape: (256, 768)
Encoding time: 0.571 seconds
Throughput: 448.45 products/second
Average L2 norm: 1.0

✓ Larger-batch benchmark passed.


## 23. Prepare the Full Product Embedding Matrix

The complete Amazon Electronics catalogue contains 1,610,012 products.

Each product will be encoded using E5-base-v2 with the `passage:` prefix.
The resulting 768-dimensional vectors are L2-normalised and written directly
to a memory-mapped matrix on disk.

A memory-mapped matrix allows the 4.6 GB embedding matrix to be written
without loading the entire matrix into RAM.

The encoding process will use batches of 256 products and can resume from the
checkpoint created previously.

In [30]:
# Open the full product embedding matrix as a memory-mapped array

EMBEDDING_SHAPE = (N_PRODUCTS, EMBEDDING_DIM)

product_embeddings = np.memmap(
    PRODUCT_EMBEDDINGS_PATH,
    dtype=np.float32,
    mode="r+",
    shape=EMBEDDING_SHAPE
)

print("Embedding matrix opened successfully.")
print("Path:", PRODUCT_EMBEDDINGS_PATH)
print("Shape:", product_embeddings.shape)
print("Dtype:", product_embeddings.dtype)
print(
    "Size on disk:",
    round(product_embeddings.nbytes / (1024**3), 2),
    "GB"
)

assert product_embeddings.shape == EMBEDDING_SHAPE
assert product_embeddings.dtype == np.float32

print("\n✓ Embedding matrix ready.")

Embedding matrix opened successfully.
Path: /users/adhp511/query-conditioned-recommender/data/generated/e5_product_embeddings.npy
Shape: (1610012, 768)
Dtype: float32
Size on disk: 4.61 GB

✓ Embedding matrix ready.


## 24. Encode the Full Amazon Electronics Catalogue

Encode every product in the full 1,610,012-product catalogue using E5-base-v2.

For each product, the representation is created from:

`passage: [product text]`

The product text contains the product title, brand/store, and category path.

The resulting 768-dimensional embedding is mean-pooled using the attention mask,
L2-normalised, and written to the memory-mapped embedding matrix.

The process is resumable. The checkpoint is updated periodically so that an
interrupted run can continue from the last completed row.

In [31]:
# Full resumable product encoding

import json
import time
import pyarrow.parquet as pq

ENCODE_BATCH_SIZE = 256
CHECKPOINT_INTERVAL = 10_000

# Reload checkpoint
with open(CHECKPOINT_PATH, "r", encoding="utf-8") as f:
    checkpoint = json.load(f)

start_row = int(checkpoint["completed_rows"])

print(f"Starting encoding from row {start_row:,} of {N_PRODUCTS:,}")

if start_row >= N_PRODUCTS:
    print("All products are already encoded.")
else:
    parquet_file = pq.ParquetFile(PRODUCT_CATALOGUE_PATH)

    processed = start_row
    last_checkpoint = start_row
    encoding_start = time.time()

    # Stream the catalogue from the beginning and skip already completed rows.
    for batch in parquet_file.iter_batches(
        batch_size=ENCODE_BATCH_SIZE,
        columns=["product_text"]
    ):
        batch_size = batch.num_rows
        batch_start = processed

        # Skip rows already completed in a previous run
        if batch_start + batch_size <= start_row:
            processed += batch_size
            continue

        # Handle a partially completed batch if necessary
        skip = max(0, start_row - batch_start)

        texts = batch.column("product_text").to_pylist()[skip:]

        if not texts:
            processed += batch_size
            continue

        actual_batch_size = len(texts)

        # E5 document/product prefix
        inputs = tokenizer(
            [f"passage: {text}" for text in texts],
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt"
        )

        inputs = {
            key: value.to(DEVICE)
            for key, value in inputs.items()
        }

        with torch.no_grad():
            outputs = e5_model(**inputs)

        # Mean pooling using the attention mask
        attention_mask = inputs["attention_mask"].unsqueeze(-1)

        masked_embeddings = outputs.last_hidden_state * attention_mask
        sum_embeddings = masked_embeddings.sum(dim=1)
        sum_mask = attention_mask.sum(dim=1).clamp(min=1)

        embeddings = sum_embeddings / sum_mask

        # L2 normalisation
        embeddings = torch.nn.functional.normalize(
            embeddings,
            p=2,
            dim=1
        )

        embeddings = embeddings.cpu().numpy().astype(np.float32)

        # Write directly to the memory-mapped matrix
        write_start = batch_start + skip
        write_end = write_start + actual_batch_size

        product_embeddings[write_start:write_end] = embeddings

        processed = write_end

        # Periodic checkpoint + disk flush
        if (
            processed - last_checkpoint >= CHECKPOINT_INTERVAL
            or processed >= N_PRODUCTS
        ):
            product_embeddings.flush()

            elapsed = time.time() - encoding_start
            rows_done = processed - start_row

            rate = rows_done / elapsed if elapsed > 0 else 0
            remaining = N_PRODUCTS - processed
            eta_seconds = remaining / rate if rate > 0 else 0

            checkpoint = {
                "completed_rows": processed,
                "total_rows": N_PRODUCTS,
                "embedding_dimension": EMBEDDING_DIM,
                "batch_size": ENCODE_BATCH_SIZE,
                "max_length": MAX_LENGTH,
                "model": "intfloat/e5-base-v2",
            }

            with open(CHECKPOINT_PATH, "w", encoding="utf-8") as f:
                json.dump(checkpoint, f, indent=2)

            last_checkpoint = processed

            print(
                f"Progress: {processed:,}/{N_PRODUCTS:,} "
                f"({processed / N_PRODUCTS * 100:.2f}%) | "
                f"Speed: {rate:.1f} products/s | "
                f"ETA: {eta_seconds / 60:.1f} min"
            )

    # Final flush
    product_embeddings.flush()

    total_elapsed = time.time() - encoding_start

    print("\n✓ Full catalogue encoding complete.")
    print(f"Products encoded: {processed:,}")
    print(f"Total time: {total_elapsed / 60:.1f} minutes")
    print(f"Average speed: {(processed - start_row) / total_elapsed:.1f} products/s")

Starting encoding from row 0 of 1,610,012
Progress: 10,240/1,610,012 (0.64%) | Speed: 346.6 products/s | ETA: 76.9 min
Progress: 20,480/1,610,012 (1.27%) | Speed: 340.5 products/s | ETA: 77.8 min
Progress: 30,720/1,610,012 (1.91%) | Speed: 341.6 products/s | ETA: 77.1 min
Progress: 40,960/1,610,012 (2.54%) | Speed: 344.4 products/s | ETA: 75.9 min
Progress: 51,200/1,610,012 (3.18%) | Speed: 343.3 products/s | ETA: 75.7 min
Progress: 61,440/1,610,012 (3.82%) | Speed: 344.8 products/s | ETA: 74.8 min
Progress: 71,680/1,610,012 (4.45%) | Speed: 342.4 products/s | ETA: 74.9 min
Progress: 81,920/1,610,012 (5.09%) | Speed: 338.8 products/s | ETA: 75.2 min
Progress: 92,160/1,610,012 (5.72%) | Speed: 335.0 products/s | ETA: 75.5 min
Progress: 102,400/1,610,012 (6.36%) | Speed: 337.4 products/s | ETA: 74.5 min
Progress: 112,640/1,610,012 (7.00%) | Speed: 336.3 products/s | ETA: 74.2 min
Progress: 122,880/1,610,012 (7.63%) | Speed: 336.3 products/s | ETA: 73.7 min
Progress: 133,120/1,610,012 (8.

## 25. Validate the Completed Product Embeddings

Check representative rows from the beginning, middle, and end of the
memory-mapped embedding matrix.

The validation confirms:

- The matrix contains the expected number of products.
- The embedding dimension is 768.
- Values are finite.
- Embeddings have the expected L2 norm of approximately 1.
- The matrix can be read correctly after the long encoding process.

In [32]:
# Validate representative embeddings from the completed matrix

CHECK_INDICES = [
    0,
    N_PRODUCTS // 2,
    N_PRODUCTS - 1
]

sample_embeddings = np.asarray(
    product_embeddings[CHECK_INDICES],
    dtype=np.float32
)

print("Sample indices:", CHECK_INDICES)
print("Sample shape:", sample_embeddings.shape)
print("Dtype:", sample_embeddings.dtype)

print("\nFinite values:", np.isfinite(sample_embeddings).all())

sample_norms = np.linalg.norm(sample_embeddings, axis=1)

print("\nL2 norms:")
for idx, norm in zip(CHECK_INDICES, sample_norms):
    print(f"Row {idx:,}: {norm:.8f}")

print("\nMinimum value:", sample_embeddings.min())
print("Maximum value:", sample_embeddings.max())

assert sample_embeddings.shape == (
    len(CHECK_INDICES),
    EMBEDDING_DIM
)

assert np.isfinite(sample_embeddings).all()

assert np.allclose(
    sample_norms,
    1.0,
    atol=1e-5
)

print("\n✓ Completed embedding matrix validation passed.")

Sample indices: [0, 805006, 1610011]
Sample shape: (3, 768)
Dtype: float32

Finite values: True

L2 norms:
Row 0: 0.99999994
Row 805,006: 0.99999994
Row 1,610,011: 0.99999994

Minimum value: -0.1620238
Maximum value: 0.09469123

✓ Completed embedding matrix validation passed.


## 26. Build the FAISS Product Index

Build a FAISS `IndexFlatIP` index over the complete Amazon Electronics
product embedding matrix.

Because all E5 product embeddings are L2-normalised, inner product is
equivalent to cosine similarity.

`IndexFlatIP` performs exact nearest-neighbour search rather than approximate
search, so the retrieved products are ranked by their exact similarity to the
query embedding.

The FAISS index will contain all 1,610,012 products in the same row order as
the product catalogue and embedding matrix.

In [33]:
# Build the exact FAISS inner-product index

print("Building FAISS index...")
print("Products:", f"{N_PRODUCTS:,}")
print("Embedding dimension:", EMBEDDING_DIM)

faiss_index = faiss.IndexFlatIP(EMBEDDING_DIM)

# Add the complete memory-mapped embedding matrix.
# FAISS requires a contiguous float32 array.
faiss_index.add(np.asarray(product_embeddings, dtype=np.float32))

print("\nFAISS index built.")
print("Index type:", type(faiss_index).__name__)
print("Number of vectors:", f"{faiss_index.ntotal:,}")
print("Dimension:", faiss_index.d)

assert faiss_index.ntotal == N_PRODUCTS
assert faiss_index.d == EMBEDDING_DIM

print("\n✓ FAISS index validation passed.")

Building FAISS index...
Products: 1,610,012
Embedding dimension: 768

FAISS index built.
Index type: IndexFlatIP
Number of vectors: 1,610,012
Dimension: 768

✓ FAISS index validation passed.


## 27. Save the Full FAISS Product Index

Persist the exact FAISS index to disk.

The FAISS row order matches the product catalogue and embedding matrix:

`FAISS row i → embedding row i → catalogue row i`

This alignment is essential because retrieved FAISS indices will later be
mapped directly back to `parent_asin`, `title`, and `product_text`.

In [34]:
# Save the complete FAISS index

FAISS_INDEX_PATH = GENERATED_DIR / "e5_faiss_product_index.bin"

faiss.write_index(
    faiss_index,
    str(FAISS_INDEX_PATH)
)

print("FAISS index saved successfully.")
print("Path:", FAISS_INDEX_PATH)
print(
    "Size:",
    round(FAISS_INDEX_PATH.stat().st_size / (1024**3), 2),
    "GB"
)
print("Vectors:", f"{faiss_index.ntotal:,}")
print("Dimension:", faiss_index.d)

assert FAISS_INDEX_PATH.exists()
assert faiss_index.ntotal == N_PRODUCTS
assert faiss_index.d == EMBEDDING_DIM

print("\n✓ FAISS index saved and validated.")

FAISS index saved successfully.
Path: /users/adhp511/query-conditioned-recommender/data/generated/e5_faiss_product_index.bin
Size: 4.61 GB
Vectors: 1,610,012
Dimension: 768

✓ FAISS index saved and validated.


## 28. Reload and Test the Saved FAISS Index

Reload the saved FAISS index from disk into a separate object and perform a
small retrieval test using a real synthetic query.

The test verifies that:

1. The persisted index can be loaded successfully.
2. It contains the expected number of products.
3. Query embeddings can be searched against the full catalogue.
4. Returned FAISS indices map correctly to catalogue rows.
5. Similarity scores are returned in descending order.

In [35]:
# Reload the saved FAISS index and test retrieval

import faiss

print("Loading FAISS index from:")
print(FAISS_INDEX_PATH)

test_faiss_index = faiss.read_index(
    str(FAISS_INDEX_PATH)
)

print("\nIndex loaded.")
print("Index type:", type(test_faiss_index).__name__)
print("Vectors:", f"{test_faiss_index.ntotal:,}")
print("Dimension:", test_faiss_index.d)

assert test_faiss_index.ntotal == N_PRODUCTS
assert test_faiss_index.d == EMBEDDING_DIM

# Load a real synthetic query
synthetic_queries = pd.read_parquet(
    GENERATED_DIR / "synthetic_queries.parquet"
)

test_query = synthetic_queries.iloc[0]["synthetic_query"]

print("\nTest query:")
print(test_query)

# Encode query using E5 query prefix
query_inputs = tokenizer(
    [f"query: {test_query}"],
    padding=True,
    truncation=True,
    max_length=MAX_LENGTH,
    return_tensors="pt"
)

query_inputs = {
    key: value.to(DEVICE)
    for key, value in query_inputs.items()
}

with torch.no_grad():
    query_outputs = e5_model(**query_inputs)

query_mask = query_inputs["attention_mask"].unsqueeze(-1)

query_embeddings = query_outputs.last_hidden_state * query_mask
query_embeddings = query_embeddings.sum(dim=1) / query_mask.sum(
    dim=1
).clamp(min=1)

query_embeddings = torch.nn.functional.normalize(
    query_embeddings,
    p=2,
    dim=1
)

query_embedding = query_embeddings.cpu().numpy().astype(np.float32)

# Search the full catalogue
TOP_K = 10

scores, indices = test_faiss_index.search(
    query_embedding,
    TOP_K
)

scores = scores[0]
indices = indices[0]

# Retrieve corresponding catalogue rows
test_results = pd.read_parquet(
    PRODUCT_CATALOGUE_PATH,
    columns=["parent_asin", "title", "store", "product_text"]
).iloc[indices].copy()

test_results.insert(0, "rank", range(1, TOP_K + 1))
test_results["similarity"] = scores

print("\nTop retrieved products:")
display(
    test_results[
        ["rank", "similarity", "parent_asin", "title", "store"]
    ]
)

# Similarity should be descending
assert np.all(scores[:-1] >= scores[1:])

# All returned indices must be valid
assert np.all(indices >= 0)
assert np.all(indices < N_PRODUCTS)

print("\n✓ Reload and retrieval test passed.")

Loading FAISS index from:
/users/adhp511/query-conditioned-recommender/data/generated/e5_faiss_product_index.bin

Index loaded.
Index type: IndexFlatIP
Vectors: 1,610,012
Dimension: 768

Test query:
looking for comfortable over-ear headphones with good bass for my ipod touch

Top retrieved products:


,rank,similarity,parent_asin,title,store
765764,1,0.872889,B00WU0OTQ4,Audio-Technica ATH-WS55IBK Solid Bass Over-Ear...,Audio-Technica
416670,2,0.868684,B07R48CHNN,1MORE Wireless Over-Ear Headphones Bluetooth C...,1MORE
800374,3,0.863618,B019266I44,"Sony 1MORE Over-Ear Headphones Bass Driven, Co...",1MORE
290822,4,0.863265,B0853NXPWJ,bopmen S80 Bluetooth Over Ear Headphones - Wir...,bopmen
1060328,5,0.862166,B08LP9BBKS,"Over Ear Headphones with Mic, Sonic The Hedgeh...",ELLASSAY
986785,6,0.861993,B001S2T58W,A4TECH SecureFit Metallic Super Bass Earphone ...,A4tech
1001226,7,0.861827,B07FQL9BLW,"Dozod Bluetooth Over Ear Headphones, Wireless ...",Dozod
1271814,8,0.860784,B09CH3HT4Z,Bluetooth Headphones with Microphone Deep Bass...,Meseto
1114312,9,0.859922,B07KQLQJ4B,NY Over Ear Best Stereo Extra Bass Portable He...,Sony
1539468,10,0.859918,B0070A9YHS,IDANCE Thick Padded Over Ear Headphones for iP...,IDANCE



✓ Reload and retrieval test passed.


## 29. Inspect the Evaluation Data

The retrieval experiment requires two pieces of information for each evaluation
example:

1. The synthetic query.
2. The held-out target product identifier (`parent_asin`).

Inspect the existing evaluation-related files to identify the exact columns and
confirm that the target products can be mapped to the full Amazon Electronics
catalogue.

No retrieval is performed in this step.

In [36]:
# Inspect the evaluation-related datasets

evaluation_files = [
    "synthetic_queries.parquet",
    "aligned_evaluation_data.parquet",
    "proposed_model_data.parquet",
]

for filename in evaluation_files:
    path = GENERATED_DIR / filename

    print("=" * 80)
    print(filename)
    print("Exists:", path.exists())

    if path.exists():
        df = pd.read_parquet(path)

        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())
        print("\nFirst 2 rows:")
        display(df.head(2))

        del df

synthetic_queries.parquet
Exists: True
Shape: (5559, 4)
Columns: ['sample_id', 'user_id', 'profile_text', 'synthetic_query']

First 2 rows:


,sample_id,user_id,profile_text,synthetic_query
0,0,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,User Preference Profile\n\nThe user has primar...,looking for comfortable over-ear headphones wi...
1,1,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,User Preference Profile\n\nThe user has primar...,looking for a reliable external hard drive for...


aligned_evaluation_data.parquet
Exists: True
Shape: (731, 12)
Columns: ['sample_id', 'user_id', 'synthetic_query', 'profile_text', 'ground_truth_asin', 'product_type', 'intended_use', 'attributes_json', 'compatibility_json', 'constraint_text', 'ground_truth_title_text', 'product_type_match']

First 2 rows:


,sample_id,user_id,synthetic_query,profile_text,ground_truth_asin,product_type,intended_use,attributes_json,compatibility_json,constraint_text,ground_truth_title_text,product_type_match
0,1,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,looking for a good quality internal hard drive...,User Preference Profile\n\nThe user has primar...,B07454F4JH,internal hard drive,backup important files,"{""quality"": ""good""}",{},Product type: internal hard drive. Attributes:...,Seagate Backup Plus Portable 5TB External Hard...,True
1,13,AE2TA5GQH4JI5RQ4W5H5PQOVYBGA,looking for a new photo backdrop stand,User Preference Profile\n\nThe user has primar...,B07VTKFLBW,photo backdrop stand,,{},{},Product type: photo backdrop stand,Mocsicka Happy Halloween Backdrop 7x5ft Vinyl ...,True


proposed_model_data.parquet
Exists: True
Shape: (5559, 9)
Columns: ['sample_id', 'user_id', 'synthetic_query', 'profile_text', 'ground_truth_asin', 'product_type', 'intended_use', 'attributes_json', 'compatibility_json']

First 2 rows:


,sample_id,user_id,synthetic_query,profile_text,ground_truth_asin,product_type,intended_use,attributes_json,compatibility_json
0,0,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,Looking for a high-speed micro sd card for my ...,User Preference Profile\n\nThe user has primar...,B007VKII6A,micro sd card,,"{""speed"": ""high-speed""}","{""device"": ""action camera""}"
1,1,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,looking for a good quality internal hard drive...,User Preference Profile\n\nThe user has primar...,B07454F4JH,internal hard drive,backup important files,"{""quality"": ""good""}",{}


## 30. Verify Ground-Truth Products in the Full Catalogue

For every evaluation example, verify that its ground-truth product identifier
(`ground_truth_asin`) is present in the full Amazon Electronics product
catalogue.

This confirms that the held-out target products are eligible to be retrieved
by the full-catalogue FAISS index.

We also check for duplicate evaluation sample IDs and missing ground-truth
identifiers.

In [37]:
# Verify evaluation targets against the full product catalogue

evaluation_data = pd.read_parquet(
    GENERATED_DIR / "proposed_model_data.parquet"
)

catalogue_ids = pd.read_parquet(
    PRODUCT_CATALOGUE_PATH,
    columns=["parent_asin"]
)["parent_asin"]

catalogue_id_set = set(catalogue_ids)

ground_truth = evaluation_data["ground_truth_asin"].astype(str)

present_mask = ground_truth.isin(catalogue_id_set)

print("Evaluation examples:", len(evaluation_data))
print("Unique sample IDs:", evaluation_data["sample_id"].nunique())
print("Missing sample IDs:", evaluation_data["sample_id"].isna().sum())

print("\nGround-truth products:")
print("Present in catalogue:", present_mask.sum())
print("Missing from catalogue:", (~present_mask).sum())

print("\nMissing ground-truth ASINs:",
      evaluation_data["ground_truth_asin"].isna().sum())

assert evaluation_data["sample_id"].nunique() == len(evaluation_data)
assert evaluation_data["ground_truth_asin"].notna().all()
assert present_mask.all()

print("\n✓ All evaluation targets are present in the full catalogue.")

Evaluation examples: 5559
Unique sample IDs: 5559
Missing sample IDs: 0

Ground-truth products:
Present in catalogue: 5559
Missing from catalogue: 0

Missing ground-truth ASINs: 0

✓ All evaluation targets are present in the full catalogue.
